In [ ]:
# GAPS
# 1. using advanced pos embeddings like RoPE
# 2. MLP -> SwiGLU/ Gated MLP
# 3. LayerNorm -> RMS Norm
# 4. Weight tying
# 5. Learning Rate Schedule - linear warmup - cosine decay
# 6. Gradient clipping
# 7. Mixed precision training
# 8. gradeint accumulation
# 9. data sharding
# 10. Exp tracking
# 11. Fault tolerance
# 12. KV caching


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import json
import os
from tqdm import tqdm

In [ ]:
# ── Environment setup ─────────────────────────────────────────────────────
# Run this cell first every Colab session (or once on a local machine).
# It mounts Drive (Colab only) and adds the repo root to sys.path so all
# project imports work correctly.
import os, sys

if os.path.exists("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

# Add the repo root to the module search path
sys.path.insert(0, os.path.abspath(".."))   # works locally from notebooks/
                                             # adjust to REPO_ROOT on Colab

import config as cfg
cfg.make_dirs()
cfg.print_config()

Mounted at /content/drive


In [ ]:
from model import GPT, GPTConfig, GPTWithKVCache
config = GPTConfig()
device = "cuda" if __import__("torch").cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
from training.data_utils import load_tokens, get_batch
from training.trainer import estimate_loss

In [ ]:
# TODO: use memmap instead of loading to disk
train_tokens = np.memmap(
    cfg.TRAIN_TOKENS,
    dtype=np.uint16,
    mode="r"
)

val_tokens = np.memmap(
    cfg.VAL_TOKENS,
    dtype=np.uint16,
    mode="r"
)

test_tokens = np.memmap(
    cfg.TEST_TOKENS,
    dtype=np.uint16,
    mode="r"
)

print("Train tokens:", len(train_tokens))
print("Val tokens:", len(val_tokens))

# train_tokens = np.fromfile(
#     cfg.TRAIN_TOKENS,
#     dtype=np.uint16
# )

# val_tokens = np.fromfile(
#     cfg.VAL_TOKENS,
#     dtype=np.uint16
# )

# test_tokens = np.fromfile(
#     cfg.TEST_TOKENS,
#     dtype=np.uint16
# )

# print("Train tokens:", len(train_tokens))

Train tokens: 455046054
Val tokens: 2180159


In [1]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.PRETRAIN_LR,
    weight_decay=cfg.PRETRAIN_WEIGHT_DECAY
)

NameError: name 'torch' is not defined

In [ ]:
checkpoint_path = cfg.PRETRAIN_CKPT

start_step = 0

if os.path.exists(checkpoint_path):

    checkpoint = torch.load(checkpoint_path)

    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])

    start_step = checkpoint["step"]

    print("Resuming from step", start_step)

Resuming from step 9000


In [ ]:
max_steps = cfg.PRETRAIN_MAX_STEPS
batch_size = cfg.PRETRAIN_BATCH_SIZE
eval_interval = cfg.PRETRAIN_EVAL_INTERVAL
save_interval = cfg.PRETRAIN_SAVE_INTERVAL

for step in range(start_step, max_steps):

    x, y = get_batch("train", batch_size)

    logits = model(x)

    loss = F.cross_entropy(
        logits.view(-1, config.vocab_size),
        y.view(-1)
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 50 == 0:
        print(f"step {step} | loss {loss.item():.4f}")

    if step % eval_interval == 0:
        val_loss = estimate_loss()
        print("validation loss:", val_loss)

    if step % save_interval == 0:

        torch.save(
            {
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "step": step
            },
            checkpoint_path
        )
torch.save(
  model.state_dict(),
  cfg.PRETRAIN_FINAL_CKPT
)

step 9000 | loss 1.9833
validation loss: 2.0287287974357606
step 9050 | loss 2.1418
step 9100 | loss 2.0537
step 9150 | loss 2.0384
step 9200 | loss 1.9537
step 9250 | loss 1.9508
step 9300 | loss 1.9128
step 9350 | loss 2.0444
step 9400 | loss 1.9894
step 9450 | loss 1.8546
step 9500 | loss 1.9884
validation loss: 2.0373305535316466
step 9550 | loss 1.8886
step 9600 | loss 1.9465
step 9650 | loss 2.0347
step 9700 | loss 2.0464
step 9750 | loss 1.9391
step 9800 | loss 2.1384
step 9850 | loss 2.0665
step 9900 | loss 2.1219
step 9950 | loss 1.8902
step 10000 | loss 1.8741
validation loss: 2.012112412452698
step 10050 | loss 1.9133
step 10100 | loss 1.7406
step 10150 | loss 1.8862
step 10200 | loss 1.9427
step 10250 | loss 2.0486
step 10300 | loss 2.0231
step 10350 | loss 1.9579
step 10400 | loss 1.8512
step 10450 | loss 2.0440
step 10500 | loss 1.9614
validation loss: 1.9685611963272094
step 10550 | loss 1.8808
step 10600 | loss 2.1349
step 10650 | loss 2.0001
step 10700 | loss 2.0542
st